# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

In [ ]:
# List all record sets and their fields
def print_all_record_sets_and_fields(ds):
    record_sets = ds.record_sets()
    print("Available record sets and fields:")
    for rs in record_sets:
        print(f"- RecordSet [@id]: {rs['@id']}, name: {rs.get('name','N/A')}")
        # List fields for this record set
        field_ids = [f['@id'] for f in ds.fields(record_set=rs['@id'])]
        print(f"  Fields @id: {field_ids}")

print_all_record_sets_and_fields(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Define available record set IDs (replace these with your findings from the previous cell)
# For this FAIR^2 dataset, the main record set is usually something like 'https://sen.science/doi/10.71728/senscience.vq4a-28xa#recordset-protein_data' or similar.
# We'll list them dynamically.

record_sets = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

for record_set in record_sets:
    print(f"Loading data for RecordSet: {record_set}")
    records = list(dataset.records(record_set=record_set))
    if records:  # Only create DataFrame if records exist
        dataframes[record_set] = pd.DataFrame(records)

# For demonstration, print the first few columns of the first non-empty record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns for first record set [{first_rs_id}]:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Choose a demo record set and field for processing
df = None
record_set_id = None

# Pick the first dataframe, if available
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

if df is not None and len(df) > 0:
    # Find a numeric field
    numeric_candidates = df.select_dtypes(include=[np.number]).columns
    if len(numeric_candidates) == 0:
        # Try to coerce float columns
        for col in df.columns:
            with np.errstate(all='ignore'):
                coerced = pd.to_numeric(df[col], errors='coerce')
                if coerced.notnull().sum() > 0:
                    df[col] = coerced
        numeric_candidates = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_candidates) > 0:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field for filtering and normalization: {numeric_field}")

        threshold = np.nanmedian(df[numeric_field]) # Median as a sample threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (median):")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Find a possible group-by field (string or categorical with few values)
        group_fields = df.select_dtypes(include=["object", "category"]).columns
        group_field = None
        for col in group_fields:
            if df[col].nunique() > 1 and df[col].nunique() < 10:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No categorical/group field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No dataframe or data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and len(df) > 0 and len(numeric_candidates) > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in Record Set: {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we have successfully loaded the FAIR^2 dataset using `mlcroissant`, explored its metadata, extracted tabular data from its main record sets using their `@id` values, and performed basic exploratory data analysis and visualizations on selected fields. You can extend this exploration by analyzing additional record sets and fields or by applying advanced analytics and visualization as required for your research or workflow.